<a href="https://colab.research.google.com/github/AntonyJohny/Innomatics_Research_Labs/blob/main/IN126001202_NLP/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 1: Conceptual Understanding
1."Love" vs "love": In NLP, these are distinct tokens due to case sensitivity (different ASCII values). Normalization is required
    so the model treats them as the same semantic concept.  

2.Stopwords: If not removed, they add noise and dimensionality without adding much meaning. However, in Sentiment Analysis,
    removing "not" can flip the meaning of a sentence entirely.  

3.Harmful Scenarios: * Sentiment Analysis: "I am not happy" $\rightarrow$ "happy" (Changes negative to positive).

4.Search/Quotes: "To be or not to be" (All words are typically stopwords; removing them leaves an empty result).Stemming vs
    Lemmatization: Stemming uses crude rules to chop suffixes (e.g., "running" $\rightarrow$ "run"), while Lemmatization uses a
    dictionary to find the base word (e.g., "better" $\rightarrow$ "good").

In [1]:
#Task 2
import re

def preprocess_text(text):
    # Task 7: Error Handling (Empty strings, only numbers, or only emojis)
    if not isinstance(text, str) or not text.strip():
        return []

    # 1. Convert text to lowercase
    # Example: "WOW!!! This is GREAT!!!" -> "wow!!! this is great!!!"
    text = text.lower()

    # 2. Remove URLs and email-like patterns
    # Example: "Visit http://example.com now" -> "visit  now"
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # 3. Handle repeated characters
    # Example: "soooo goooood!!!" -> "so good"
    # This regex finds a character followed by itself and replaces the whole group with one instance
    text = re.sub(r'(.)\1+', r'\1', text)

    # 4. Remove numbers
    # Example: "I have 2 dogs" -> "i have  dogs"
    text = re.sub(r'\d+', '', text)

    # 5. Remove special characters (to clean up the "!!!" from the examples)
    text = re.sub(r'[^a-z\s]', '', text)

    # 6. Remove extra spaces
    # Example: "This   is      good" -> "this is good"
    tokens = text.split()

    # 7. Remove very short tokens (length <= 2)
    # Exception: keep "no", "not"
    keep_exceptions = {'no', 'not'}
    tokens = [t for t in tokens if len(t) > 2 or t in keep_exceptions]

    return tokens

# Verification of specific Assignment Examples:
print(f"Numbers: {preprocess_text('I have 2 dogs')}")
print(f"Repeated: {preprocess_text('soooo goooood!!!')}")
print(f"Lowercase: {preprocess_text('WOW!!! This is GREAT!!!')}")
print(f"URL: {preprocess_text('Visit http://example.com now')}")

Numbers: ['have', 'dogs']
Repeated: ['god']
Lowercase: ['wow', 'this', 'great']
URL: ['visit', 'now']


In [2]:
#Task 3
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print(f"{'Original Text':<40} | {'Cleaned Sentence'}")
print("-" * 70)

for sentence in sample_inputs:
    tokens = preprocess_text(sentence)
    clean_sentence = " ".join(tokens)
    print(f"{sentence:<40} | {clean_sentence}")

Original Text                            | Cleaned Sentence
----------------------------------------------------------------------
Get 100% FREE access now!!!              | get fre aces now
I absolutely looooved this product 😍😍    | absolutely loved this product
Worst service ever... 0/10               | worst service ever
Call me at 9876543210                    | cal
This is THE best course!!!               | this the best course
Visit https://openai.com now!            | visit now
Nooooo this is baaad!!!                  | no this bad
OK OK OK I got it                        | got
Win $$$ now!!! Limited offer!!!          | win now limited ofer
I am not happy with this                 | not hapy with this


In [3]:
#Task 4
results = []

for sentence in sample_inputs:
    tokens = preprocess_text(sentence)

    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_len = sum(len(t) for t in tokens) / total_tokens if total_tokens > 0 else 0

    results.append({
        "Original": sentence,
        "Tokens": tokens,
        "Cleaned": " ".join(tokens),
        "Total": total_tokens,
        "Unique": unique_tokens,
        "Avg_Len": round(avg_len, 2)
    })

# Convert to DataFrame for clear viewing
import pandas as pd
df_analytics = pd.DataFrame(results)
print(df_analytics[['Original', 'Total', 'Unique', 'Avg_Len']])

# Analysis Answers:
# 1. Sentence with most noise: "Visit https://openai.com now!" (URL and symbols removed)
# 2. Most meaningful: "I am not happy with this" (Preserved 'not' and 'happy')

                                Original  Total  Unique  Avg_Len
0            Get 100% FREE access now!!!      4       4     3.25
1  I absolutely looooved this product 😍😍      4       4     6.50
2             Worst service ever... 0/10      3       3     5.33
3                  Call me at 9876543210      1       1     3.00
4             This is THE best course!!!      4       4     4.25
5          Visit https://openai.com now!      2       2     4.00
6                Nooooo this is baaad!!!      3       3     3.00
7                      OK OK OK I got it      1       1     3.00
8        Win $$$ now!!! Limited offer!!!      4       4     4.25
9               I am not happy with this      4       4     3.75


In [4]:
#Task 5
from collections import Counter

# Combine all lists of tokens into one master list
all_tokens = [token for sublist in results for token in sublist['Tokens']]

word_counts = Counter(all_tokens)

print("Top 10 Most Frequent Words:", word_counts.most_common(10))
print("Top 5 Least Frequent Words:", word_counts.most_common()[:-6:-1])

Top 10 Most Frequent Words: [('this', 4), ('now', 3), ('get', 1), ('fre', 1), ('aces', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Top 5 Least Frequent Words: [('with', 1), ('hapy', 1), ('not', 1), ('ofer', 1), ('limited', 1)]


In [5]:
#Task 6
def full_pipeline(text_list):
    cleaned_sentences = []
    all_tokens = []

    for text in text_list:
        tokens = preprocess_text(text)
        all_tokens.extend(tokens)
        cleaned_sentences.append(" ".join(tokens))

    return {
        "tokens": all_tokens,
        "clean_sentences": cleaned_sentences
    }

# Execute
pipeline_output = full_pipeline(sample_inputs)
print(pipeline_output)

{'tokens': ['get', 'fre', 'aces', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'cal', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'ofer', 'not', 'hapy', 'with', 'this'], 'clean_sentences': ['get fre aces now', 'absolutely loved this product', 'worst service ever', 'cal', 'this the best course', 'visit now', 'no this bad', 'got', 'win now limited ofer', 'not hapy with this']}


In [6]:
#Task 7
test_cases = ["", "😊😊😊", "12345 678"]

for case in test_cases:
    print(f"Input: '{case}' -> Cleaned: {preprocess_text(case)}")

Input: '' -> Cleaned: []
Input: '😊😊😊' -> Cleaned: []
Input: '12345 678' -> Cleaned: []
